# 05 · RAG index: split the documents into passages and embed them

Prepares the documents' text (from `03_documents.ipynb`) for the assistant in the app, which answers questions from
both the structured data and the documents.

1. **Split** each extracted document into passages of ~1,200 characters (with a 200-character overlap, cut at paragraph
   or sentence ends).
2. **Embed** each passage with a local multilingual model (`paraphrase-multilingual-mpnet-base-v2`, via
   `sentence-transformers`; free, no API key, ~1 GB downloaded once per session).
3. **Store** passages and vectors in `procurement.duckdb` (`rag_chunks`), plus which contracts and files contain each text
   (`rag_doc_map`), and build a BM25 keyword index for exact terms such as clause numbers, NIFs or CPV codes.

**Duplicates:** the same document (for example a framework agreement's specifications) is often attached to many
contracts. Texts are keyed by their MD5 hash, so each distinct text is embedded only once.

**Incremental:** only texts not indexed yet are processed, so the daily run after `03` is quick.
The splitting and embedding code lives in `app/rag.py`, the same module the app uses to embed questions.

**Use a GPU runtime in Colab:** *Runtime → Change runtime type → T4 GPU*. On the free CPU the model manages about
1 passage per second (hours for the full set); on the T4 GPU, hundreds per second (a few minutes).
Indexing a few new documents on a normal day is fine on CPU.

**Runs in Google Colab or locally.** The first code cell detects the environment.

In [1]:
# ── Environment: Google Colab or local (VS Code / Jupyter) ─────────────────────────────
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_FOLDER = "procurement-poc"   # folder in "My Drive" that holds the project data
PIP_PACKAGES = ["duckdb>=1.1", "sentence-transformers>=3.0"]

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive") / DRIVE_FOLDER / "data"   # persistent: survives the session
    WORK = Path("/content/work")                                     # fast local disk: wiped when the session ends
    if PIP_PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES], check=True)
else:
    DATA = Path("data")
    WORK = DATA

DATA.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print(f"running in Colab: {IN_COLAB} | data folder: {DATA}")


# DuckDB is slow and can corrupt on a network drive, so in Colab the notebook works on a local copy
# of the database and copies it back to Drive with push_db(). Locally both paths are the same file.
DB_NAME = "procurement.duckdb"
DB_FILE = WORK / DB_NAME


def pull_db():
    """Colab: copy the database from Drive to local disk before connecting."""
    src = DATA / DB_NAME
    if WORK != DATA and src.exists():
        shutil.copy2(src, DB_FILE)
        print(f"database copied from Drive ({src.stat().st_size / 1e6:.1f} MB)")


def push_db(con=None):
    """Colab: save the local copy back to Drive. With an open connection, it checkpoints first."""
    if WORK == DATA:
        return
    if con is not None:
        con.execute("CHECKPOINT")
    tmp = DATA / (DB_NAME + ".tmp")
    shutil.copy2(DB_FILE, tmp)
    tmp.replace(DATA / DB_NAME)
    print(f"database saved to Drive ({DB_FILE.stat().st_size / 1e6:.1f} MB)")

Mounted at /content/drive
running in Colab: True | data folder: /content/drive/MyDrive/procurement-poc/data


In [2]:
import time

import duckdb
import pandas as pd

APP = (DATA.parent / "app").resolve()          # app/ next to data/ (in Drive: My Drive/procurement-poc/app)
missing = [f for f in ["rag.py", "lib.py", "app.py"] if not (APP / f).exists()]
assert not missing, (f"{APP} is missing {missing}. Upload the latest app/ folder to Drive (replacing the old one), "
                     "then Runtime -> Restart session and run again.")
sys.path.insert(0, str(APP))
import rag                                      # noqa: E402

LIMIT_TEXTS = None      # distinct texts to index per run; None = all pending
PUSH_EVERY_S = 300      # Colab: save the database to Drive every 5 minutes

import torch  # noqa: E402

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    BATCH_CHUNKS = 2048
else:
    print("No GPU: indexing runs at ~1 passage/s. For the first full run use Runtime -> Change runtime type -> T4 GPU.")
    BATCH_CHUNKS = 128

pull_db()
con = duckdb.connect(str(DB_FILE))
con.execute("SET memory_limit = '3GB'")     # leave room for the embedding model in Colab's ~12 GB
con.execute(rag.SCHEMA)

meta = dict(con.execute("SELECT key, value FROM rag_meta").fetchall())
settings = {"embed_model": rag.EMBED_MODEL, "embed_backend": rag.EMBED_BACKEND, "embed_dim": str(rag.EMBED_DIM),
            "chunk_chars": str(rag.CHUNK_CHARS), "overlap_chars": str(rag.OVERLAP_CHARS)}
if meta and {k: meta.get(k) for k in settings} != settings:
    # vectors from another model/library can't be mixed with new ones: start the index again
    print(f"index settings changed ({meta} -> {settings}): rebuilding the index from scratch")
    try:
        con.execute("INSTALL fts; LOAD fts; PRAGMA drop_fts_index('rag_chunks')")
    except Exception:
        pass
    con.execute("DROP TABLE IF EXISTS rag_chunks; DROP TABLE IF EXISTS rag_doc_map; DROP TABLE IF EXISTS rag_meta")
    con.execute(rag.SCHEMA)
con.executemany("INSERT OR REPLACE INTO rag_meta VALUES (?, ?)", list(settings.items()))
print("index settings:", settings)

No GPU: indexing runs at ~1 passage/s. For the first full run use Runtime -> Change runtime type -> T4 GPU.
database copied from Drive (162.0 MB)
index settings: {'embed_model': 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2', 'embed_backend': 'sentence-transformers', 'embed_dim': '768', 'chunk_chars': '1200', 'overlap_chars': '200'}


## 1. Which texts need indexing
`rag_doc_map` is rebuilt from `documents` on every run (it's cheap), so new contracts that reuse an already indexed text
are linked to it without embedding anything again.

In [3]:
con.execute("""
    CREATE OR REPLACE TABLE rag_doc_map AS
    SELECT DISTINCT md5(text) AS text_hash, contract_id, file_name
    FROM documents
    WHERE text IS NOT NULL AND length(trim(text)) > 0 AND status = 'extracted'
""")
# passages of texts that no longer exist (a document was re-extracted and changed)
removed_hashes = con.execute("""
    SELECT DISTINCT text_hash FROM rag_chunks WHERE text_hash NOT IN (SELECT text_hash FROM rag_doc_map)
""").fetchall()
con.execute("DELETE FROM rag_chunks WHERE text_hash NOT IN (SELECT text_hash FROM rag_doc_map)")

pending = con.execute("""
    SELECT md5(text) AS text_hash, ANY_VALUE(text) AS text
    FROM documents
    WHERE text IS NOT NULL AND length(trim(text)) > 0 AND status = 'extracted'
      AND md5(text) NOT IN (SELECT DISTINCT text_hash FROM rag_chunks)
    GROUP BY 1
""").df()
if LIMIT_TEXTS:
    pending = pending.head(LIMIT_TEXTS)

stats = con.execute("""
    SELECT COUNT(*) AS documents, COUNT(DISTINCT text_hash) AS distinct_texts, COUNT(DISTINCT contract_id) AS contracts
    FROM rag_doc_map
""").df()
print(f"{len(pending)} distinct texts to index")
stats

0 distinct texts to index


,documents,distinct_texts,contracts
0,920,702,290


## 2. Split, embed and store
The first call downloads the embedding model (~1 GB). Speed: ~1 passage/s on Colab's free CPU, several hundred per
second on a T4 GPU. Each batch is committed and the database is saved to Drive every 5 minutes, so an interrupted run
keeps what it already did.

In [4]:
next_id = con.execute("SELECT COALESCE(MAX(chunk_id), 0) + 1 FROM rag_chunks").fetchall()[0][0]
start = last_push = time.time()
done_texts = done_chunks = 0


def flush(rows):
    global next_id
    vectors = rag.embed([r["text"] for r in rows])
    batch = pd.DataFrame(rows)
    batch["chunk_id"] = range(next_id, next_id + len(batch))
    batch["embedding"] = list(vectors)
    next_id += len(batch)
    con.execute("BEGIN TRANSACTION")
    con.execute(f"""INSERT INTO rag_chunks
                    SELECT chunk_id, text_hash, chunk_no, text, embedding::FLOAT[{rag.EMBED_DIM}] FROM batch""")
    con.execute("COMMIT")


buffer = []
for i, (text_hash, text) in enumerate(pending.itertuples(index=False), 1):
    buffer += [{"text_hash": text_hash, "chunk_no": n, "text": t} for n, t in enumerate(rag.chunk_text(text))]
    done_texts += 1
    if len(buffer) >= BATCH_CHUNKS or i == len(pending):     # whole documents per batch, never half of one
        if buffer:
            flush(buffer)
            done_chunks += len(buffer)
            buffer = []
        rate = done_chunks / max(time.time() - start, 1e-9)
        print(f"{done_texts}/{len(pending)} texts, {done_chunks:,} passages ({rate:.0f}/s)")
        if time.time() - last_push > PUSH_EVERY_S:
            push_db(con)
            last_push = time.time()

print(f"done in {time.time() - start:.0f}s")

done in 0s


## 3. Keyword index (BM25)
DuckDB's `fts` extension, with Portuguese stemming and accents removed. Rebuilt from scratch (fast).
If the extension can't be installed, the app falls back to a simple term match.

In [5]:
try:
    con.execute("INSTALL fts; LOAD fts")
    fts_exists = con.execute("SELECT COUNT(*) FROM information_schema.schemata "
                             "WHERE schema_name = 'fts_main_rag_chunks'").fetchall()[0][0] > 0
    if fts_exists and done_chunks == 0 and not len(removed_hashes):
        raise StopIteration("nothing new since the last run")
    con.execute("""PRAGMA create_fts_index('rag_chunks', 'chunk_id', 'text',
                   stemmer = 'portuguese', stopwords = 'none', strip_accents = 1, lower = 1, overwrite = 1)""")
    print("BM25 index built")
except StopIteration as e:
    print("BM25 index up to date:", e)
except Exception as e:
    print("BM25 index not built:", e)
push_db(con)

BM25 index up to date: nothing new since the last run
database saved to Drive (162.0 MB)


## 4. Try a search
The same hybrid search the assistant uses: semantic (vectors) + keywords (BM25), merged with Reciprocal Rank Fusion.
Optionally limit it to one contract.

In [6]:
QUESTION = "prazo de garantia dos bens fornecidos"
CONTRACT_ID = None          # e.g. 15645692

for hit in rag.search(con, QUESTION, contract_id=CONTRACT_ID, k=5):
    where = "; ".join(f"{c} · {f}" for c, f in hit["sources"][:3]) + (" ..." if len(hit["sources"]) > 3 else "")
    print(f"score {hit['score']}  |  {where}")
    print("   ", hit["text"][:400].replace("\n", " "), "\n")

a carregar o modelo de pesquisa sentence-transformers/paraphrase-multilingual-mpnet-base-v2 (cpu)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

modelo de pesquisa carregado


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

score 0.0314  |  14562576 · Processo Concurso/1_Caderno_de_Encargos_AD_42_2026.pdf
    prazo acordado, a(s) fatura(s) ou documento(s) equivalente(s), emitida(s) pelo adjudicatário/cocontratante; b) Verificar o cumprimento das obrigações previstas para o adjudicatário/cocontratante.  Cláusula 9.ª Garantia 1. O adjudicatário/cocontratante garantirá, sem qualquer encargo para a entidade adjudicante/contraente público, os bens fornecidos pelo prazo de garantia mínimo de 36 meses, contra 

score 0.0225  |  15234586 · Processo Concurso/1_Caderno_de_Encagos_CP_n_05_SASUBI_2026_pdf._signed.pdf; 15234466 · Processo Concurso/1_Caderno_de_Encagos_CP_n_05_SASUBI_2026_pdf._signed.pdf; 15234772 · Processo Concurso/1_Caderno_de_Encagos_CP_n_05_SASUBI_2026_pdf._signed.pdf
    48 horas, qualquer discordância quanto aos bens fornecidos, valores faturados e respetiva fundamentação; vi. Pagar, conforme decorre da prescrição do artigo 299.º do CCP, as faturas emitidas pelo adjudicatário/cocontratante, se c

In [7]:
con.sql("""
    SELECT COUNT(DISTINCT text_hash) AS texts, COUNT(*) AS passages,
           ROUND(AVG(length(text))) AS avg_chars,
           (SELECT COUNT(DISTINCT contract_id) FROM rag_doc_map) AS contracts_covered
    FROM rag_chunks
""").df()

,texts,passages,avg_chars,contracts_covered
0,702,18882,1037.0,290


In [8]:
con.close()
push_db()   # Colab: copy the database back to Drive

database saved to Drive (162.0 MB)
